# YOLO Training Experiments

This notebook records the model-training experiments for the retail product detector.

**Experiment sequence**
1. Verify the Colab GPU environment and prepare the processed dataset.
2. Run a one-epoch YOLO11n smoke test.
3. Train the YOLO11n baseline.
4. Test the hypothesis that additional model capacity improves visually similar grocery classes by training YOLO11s with the same core settings.
5. Evaluate the selected YOLO11s model once on the held-out test split.
6. Export the final model to ONNX and compare one validation prediction against PyTorch.

Repeated setup cells, failed archive/download attempts, and temporary Colab recovery steps have been removed from this cleaned notebook. Full plots and CSV artifacts are committed under `results/`.


## 1. Environment


In [ ]:
!pip install -q ultralytics==8.4.130 gdown onnxruntime


In [ ]:
import torch
import ultralytics

print("Ultralytics:", ultralytics.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


Training was run on a **Tesla T4** using **Ultralytics 8.4.130**. Local preprocessing, evaluation, serving, and CPU benchmarking were performed separately in the repository environment.


## 2. Load the processed dataset


In [ ]:
!gdown --fuzzy "https://drive.google.com/file/d/1MAUSH06nXjBy1kPP2kdqLTBjT5I5qUjS/view?usp=drive_link" \
    -O /content/groceries-yolo.tar.gz

In [ ]:
!mkdir -p /content/retail-data
!tar -xzf /content/groceries-yolo.tar.gz -C /content/retail-data

In [ ]:
from pathlib import Path

YOLO_ROOT = Path("/content/retail-data/yolo")

for split in ["train", "val", "test"]:
    n_images = len(list((YOLO_ROOT / "images" / split).glob("*")))
    n_labels = len(list((YOLO_ROOT / "labels" / split).glob("*.txt")))
    print(split, n_images, n_labels)

train 3957 3957
val 495 495
test 495 495


In [ ]:
import yaml

class_names = [
    "beans",
    "cake",
    "candy",
    "cereal",
    "chips",
    "chocolate",
    "coffee",
    "corn",
    "fish",
    "flour",
    "honey",
    "jam",
    "juice",
    "milk",
    "nuts",
    "oil",
    "pasta",
    "rice",
    "soda",
    "spices",
    "sugar",
    "tea",
    "tomato_sauce",
    "vinegar",
    "water",
]

dataset_config = {
    "path": "/content/retail-data/yolo",
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": {i: name for i, name in enumerate(class_names)},
}

with open("/content/groceries.yaml", "w") as f:
    yaml.safe_dump(dataset_config, f, sort_keys=False)

print(open("/content/groceries.yaml").read())

path: /content/retail-data/yolo
train: images/train
val: images/val
test: images/test
names:
  0: beans
  1: cake
  2: candy
  3: cereal
  4: chips
  5: chocolate
  6: coffee
  7: corn
  8: fish
  9: flour
  10: honey
  11: jam
  12: juice
  13: milk
  14: nuts
  15: oil
  16: pasta
  17: rice
  18: soda
  19: spices
  20: sugar
  21: tea
  22: tomato_sauce
  23: vinegar
  24: water



The processed split contains **3,957 train**, **495 validation**, and **495 test** images across 25 classes.


## 3. Smoke test — YOLO11n


In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")

In [ ]:
smoke_results = model.train(
    data="/content/groceries.yaml",
    epochs=1,
    imgsz=320,
    batch=32,
    device=0,
    workers=2,
    project="/content/runs",
    name="smoke_test",
    seed=42,
)

**Recorded validation result after 1 epoch**

| Precision | Recall | mAP@50 | mAP@50:95 |
|---:|---:|---:|---:|
| 0.141 | 0.227 | 0.115 | 0.087 |

The smoke test was used only to verify that the dataset, labels, GPU training loop, and validation pipeline worked end to end.


## 4. Baseline — YOLO11n


In [ ]:
from ultralytics import YOLO

baseline_model = YOLO("yolo11n.pt")

baseline_results = baseline_model.train(
    data="/content/groceries.yaml",
    epochs=50,
    imgsz=224,
    batch=32,
    device=0,
    workers=2,
    seed=42,
    patience=10,
    project="/content/runs",
    name="baseline_yolo11n_224",
    plots=True,
)

**Baseline validation result**

| Model | Precision | Recall | mAP@50 | mAP@50:95 |
|---|---:|---:|---:|---:|
| YOLO11n | 0.845 | 0.834 | 0.892 | 0.754 |

Training configuration: 50 epochs, 224×224 input, batch size 32, seed 42, COCO-pretrained weights.

![YOLO11n training curves](../results/yolo11n_baseline/results.png)

![YOLO11n normalized confusion matrix](../results/yolo11n_baseline/confusion_matrix_normalized.png)


### Baseline observation

The baseline converged cleanly, but several visually similar or harder categories remained weaker. This motivated a controlled capacity experiment rather than changing several hyperparameters at once.


## 5. Controlled capacity experiment — YOLO11s


In [ ]:
from ultralytics import YOLO

model_s = YOLO("yolo11s.pt")

results_s = model_s.train(
    data="/content/groceries.yaml",
    epochs=50,
    imgsz=224,
    batch=32,
    device=0,
    workers=2,
    seed=42,
    patience=10,
    project="/content/runs",
    name="experiment1_yolo11s_224",
    plots=True,
)

The model size was the primary experimental change; the dataset split, image size, epochs, batch size, and seed were kept fixed.

| Model | Precision | Recall | mAP@50 | mAP@50:95 |
|---|---:|---:|---:|---:|
| YOLO11n | 0.845 | 0.834 | 0.892 | 0.754 |
| **YOLO11s** | **0.872** | **0.879** | **0.939** | **0.820** |

YOLO11s improved validation mAP@50:95 by **0.066**, so it was selected as the final model.

![YOLO11s training curves](../results/yolo11s_final/results.png)

![YOLO11s normalized confusion matrix](../results/yolo11s_final/confusion_matrix_normalized.png)


## 6. Final held-out test evaluation


In [ ]:
from ultralytics import YOLO

final_model = YOLO(
    "/content/runs/experiment1_yolo11s_224/weights/best.pt"
)

In [ ]:
test_results = final_model.val(
    data="/content/groceries.yaml",
    split="test",
    imgsz=224,
    batch=32,
    device=0,
    plots=True,
    project="/content/runs",
    name="final_test_yolo11s_224"
)

The test split was kept out of model selection and evaluated after choosing YOLO11s.

| Precision | Recall | mAP@50 | mAP@50:95 |
|---:|---:|---:|---:|
| **0.897** | **0.849** | **0.917** | **0.797** |

![Final test normalized confusion matrix](../results/final_test/confusion_matrix_normalized.png)


## 7. ONNX export


In [ ]:
from ultralytics import YOLO

final_model = YOLO(
    "/content/runs/experiment1_yolo11s_224/weights/best.pt"
)

In [ ]:
onnx_path = final_model.export(
    format="onnx",
    imgsz=224,
    simplify=True,
    opset=17
)

print(onnx_path)

In [ ]:
from pathlib import Path

onnx_path = Path(onnx_path)

print("Exists:", onnx_path.exists())
print("Size MB:", onnx_path.stat().st_size / (1024 * 1024))

Exists: True
Size MB: 36.06580352783203


The final YOLO11s checkpoint was exported to ONNX with a 224×224 input and opset 17. Model binaries are intentionally excluded from Git history.


## 8. PyTorch vs. ONNX prediction check


In [ ]:
onnx_model = YOLO(str(onnx_path))

In [ ]:
from pathlib import Path

sample_image = next(
    Path("/content/retail-data/yolo/images/val").glob("*")
)

print(sample_image)

/content/retail-data/yolo/images/val/CORN0062.png


In [ ]:
pt_result = final_model.predict(
    source=str(sample_image),
    imgsz=224,
    conf=0.25,
    verbose=False
)[0]

In [ ]:
onnx_result = onnx_model.predict(
    source=str(sample_image),
    imgsz=224,
    conf=0.25,
    verbose=False
)[0]

In [ ]:
print("PyTorch detections:", len(pt_result.boxes))
print("ONNX detections:", len(onnx_result.boxes))


PyTorch detections: 10
ONNX detections: 10


In [ ]:
import numpy as np

pt_classes = pt_result.boxes.cls.cpu().numpy()
onnx_classes = onnx_result.boxes.cls.cpu().numpy()

pt_conf = pt_result.boxes.conf.cpu().numpy()
onnx_conf = onnx_result.boxes.conf.cpu().numpy()

pt_boxes = pt_result.boxes.xyxy.cpu().numpy()
onnx_boxes = onnx_result.boxes.xyxy.cpu().numpy()

print("Same detection count:", len(pt_boxes) == len(onnx_boxes))
print("Same class IDs:", np.array_equal(pt_classes, onnx_classes))
print("Max confidence difference:", float(np.max(np.abs(pt_conf - onnx_conf))))
print("Max box-coordinate difference:", float(np.max(np.abs(pt_boxes - onnx_boxes))))


On the checked validation sample, PyTorch and ONNX produced the same **10 detections**, with matching class IDs, confidence scores, and box coordinates to the displayed precision in the original run. This is a sample-level export sanity check, not a claim of bitwise equivalence over the full dataset.


## Experiment summary

| Stage | Model / condition | mAP@50 | mAP@50:95 |
|---|---|---:|---:|
| Smoke test | YOLO11n, 1 epoch | 0.115 | 0.087 |
| Baseline validation | YOLO11n | 0.892 | 0.754 |
| Final-model validation | YOLO11s | 0.939 | 0.820 |
| Held-out test | YOLO11s | **0.917** | **0.797** |

The repository also contains separate scripts for final evaluation, CPU benchmarking, FastAPI inference, example generation, and the later mixed-class co-occurrence stress test.
